Loading dataset

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download("kaustubhdikshit/neu-surface-defect-database")
dataset_path = os.path.join(path, "NEU-DET")

print(dataset_path)
print(os.listdir(dataset_path))

	Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import os

Combining All Images

In [ ]:
import os
import shutil

base_path = dataset_path  # your NEU-DET path

train_dir = os.path.join(base_path, "train", "images")
val_dir = os.path.join(base_path, "validation", "images")

# Change all_dir to a writable location
all_dir = os.path.join("/kaggle/working/", "NEU-DET_all_images")

os.makedirs(all_dir, exist_ok=True)

classes = os.listdir(train_dir)

for cls in classes:
    os.makedirs(os.path.join(all_dir, cls), exist_ok=True)

    # Move train images
    train_class_dir = os.path.join(train_dir, cls)
    for img in os.listdir(train_class_dir):
        shutil.copy(
            os.path.join(train_class_dir, img),
            os.path.join(all_dir, cls, img)
        )

    # Move validation images
    val_class_dir = os.path.join(val_dir, cls)
    for img in os.listdir(val_class_dir):
        shutil.copy(
            os.path.join(val_class_dir, img),
            os.path.join(all_dir, cls, img)
        )

print("All images combined successfully.")

Splitting the data into train and validation

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    all_dir,
    validation_split=0.3,
    subset="training",
    seed=42,
    image_size=(128,128),
    batch_size=64
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    all_dir,
    validation_split=0.3,
    subset="validation",
    seed=42,
    image_size=(128,128),
    batch_size=64
)

# SAVE CLASS NAMES HERE
class_names = train_ds.class_names
print("Classes:", class_names)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

DATA AGUMENTATION

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
])

BASELINE MODEL

In [ ]:
def build_baseline_model(num_classes=6):

    inputs = tf.keras.Input(shape=(128,128,3))
    x = tf.keras.layers.Rescaling(1./255)(inputs)

    x = tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    outputs = tf.keras.layers.Dense(num_classes)(x)

    model = tf.keras.Model(inputs, outputs)

    return model

CUSTOM BUILD MODEL

In [ ]:
def build_regularized_model(num_classes=6):
    inputs = keras.Input(shape=(128,128,3))

    x = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.03),
    ])(inputs)

    x = layers.Rescaling(1./255)(x)

    x = layers.Conv2D(32,3,padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64,3,padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128,3,padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(num_classes)(x)

    return keras.Model(inputs, outputs)

MODEL COMPILATION

In [ ]:
model = build_baseline_model()

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0003),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
model_reg = build_regularized_model()

model_reg.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0003),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

MODEL TRAONING

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)
history_reg = model_reg.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[early_stop]
)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30
)

In [ ]:
trained_model = model

ACCURACY PLOT

In [ ]:
plt.plot(history.history['val_accuracy'], label='Baseline')
plt.plot(history_reg.history['val_accuracy'], label='Regularized')
plt.legend()
plt.title("Validation Accuracy Comparison")
plt.show()

CUSTOM CNN CONFUSION MATRIX

In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np
import seaborn as sns

y_true = []
y_pred = []

for images, labels in val_ds:
    logits = model_reg.predict(images)
    preds = np.argmax(logits, axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(preds)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names,
            yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
def plot_confusion_matrix(model, dataset, class_names):
    from sklearn.metrics import confusion_matrix
    import numpy as np
    import seaborn as sns
    import matplotlib.pyplot as plt

    y_true = []
    y_pred = []

    for images, labels in dataset:
        logits = model.predict(images)
        preds = np.argmax(logits, axis=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds)

    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names,
                yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.show()

    return cm

BASELINE CNN CONFUSION MATRIX

In [ ]:
class_names = sorted(os.listdir(all_dir))
cm_baseline = plot_confusion_matrix(model, val_ds, class_names)

In [ ]:
def show_misclassified(model, dataset, class_names, max_images=10):
    import matplotlib.pyplot as plt
    import numpy as np

    shown = 0

    for images, labels in dataset:
        logits = model.predict(images)
        preds = np.argmax(logits, axis=1)

        for i in range(len(images)):
            if preds[i] != labels[i]:
                plt.imshow(images[i].numpy().astype("uint8"))
                plt.title(
                    f"True: {class_names[int(labels[i])]} | "
                    f"Pred: {class_names[int(preds[i])]}"
                )
                plt.axis("off")
                plt.show()

                shown += 1
                if shown >= max_images:
                    return

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import cv2


# 1️ Prepareing Image


IMG_SIZE = 128

img_path = "/kaggle/working/NEU-DET_all_images/crazing/crazing_62.jpg"

img = tf.keras.utils.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
img_array = tf.keras.utils.img_to_array(img)

# If you used Rescaling(1./255) inside model -- DO NOT divide again
img_array_expanded = np.expand_dims(img_array, axis=0)


# 2️ Ensureing Model Is Built


_ = trained_model(tf.zeros((1, IMG_SIZE, IMG_SIZE, 3)))


# 3️ Automatically Finding Last Conv Layer


last_conv_layer = None
for layer in reversed(trained_model.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer.name
        break

print("Using last conv layer:", last_conv_layer)


# 4️ Grad-CAM Function


def make_gradcam_heatmap(img_array, model, last_conv_layer_name, class_index=None):

    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)

        if class_index is None:
            class_index = tf.argmax(predictions[0])

        loss = predictions[:, class_index]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap) + 1e-8

    return heatmap.numpy(), predictions


# 5️ Generate Heatmap


heatmap, predictions = make_gradcam_heatmap(
    img_array_expanded,
    trained_model,
    last_conv_layer
)

pred_class = tf.argmax(predictions[0]).numpy()
confidence = tf.reduce_max(tf.nn.softmax(predictions[0])).numpy()

print("Predicted class:", pred_class)
print("Confidence:", confidence)


# 6️ Overlay Heatmap Properly


heatmap = np.uint8(255 * heatmap)
heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
heatmap = cv2.GaussianBlur(heatmap, (9,9), 0)   # smoother
heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

overlay = cv2.addWeighted(
    img_array.astype(np.uint8),
    0.6,
    heatmap,
    0.4,
    0
)


# 7️ Display Results


plt.figure(figsize=(12,5))

plt.subplot(1,3,1)
plt.imshow(img_array.astype(np.uint8))
plt.title("Original")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(heatmap)
plt.title("Heatmap")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(overlay)
plt.title("Grad-CAM Overlay")
plt.axis("off")

plt.show()

In [ ]:
print("Baseline parameters:", model.count_params())
print("Custom model parameters:", model_reg.count_params())

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.legend(['Train','Validation'])
plt.title("Accuracy")
plt.show()

plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.legend(['Train','Validation'])
plt.title("Loss")
plt.show()

In [ ]:
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.utils import load_img, img_to_array

img_path = "/kaggle/working/NEU-DET_all_images/crazing/crazing_62.jpg"

img = load_img(img_path, target_size=(128,128))
img_array = img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)

logits = model.predict(img_array)
probs = tf.nn.softmax(logits)

pred_class = np.argmax(probs)
confidence = np.max(probs)

# get class names from folder
class_names = sorted(os.listdir(all_dir))

print("Predicted class:", class_names[pred_class])
print("Confidence:", float(confidence))

In [ ]:
from tensorflow.keras.utils import plot_model

plot_model(
    model_reg,
    to_file="model_architecture.png",
    show_shapes=True,
    show_layer_names=True
)